# 06 - Production Pipeline Check
This notebook contains **no feature engineering and no training logic**. It only uses the production package:
* `src/features.py` - single source of truth for feature engineering
* `src/train.py` - frozen training (run once: `python -m src.train`)
* `src/predict.py` - public interface `predict_laptop_price(specs)`
* `artifacts/` - saved pipeline + metadata

It shows: the saved model's metadata, one worked example, an in-sample sanity check on 11 representative laptops,
and a prediction made from a fresh Python process that has only `src/` and `artifacts/`.

In [1]:
import json
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import pandas as pd

from src.features import specs_from_raw
from src.predict import load_artifacts, predict_laptop_price
from src.train import load_clean_raw, MODEL_PATH, METADATA_PATH

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 45)

## 1. Saved artifacts

In [2]:
pipeline, meta = load_artifacts()
print('Model file:   ', MODEL_PATH.name, f'({MODEL_PATH.stat().st_size / 1024:.0f} KB)')
print('Metadata file:', METADATA_PATH.name)
for k in ['model_name', 'model_version', 'training_date', 'target_transformation', 'price_inverse_transformation']:
    print(f'{k:<30}', meta[k])
print(f"{'training observations':<30}", meta['training_data']['n_training_observations'])
print(f"{'rare companies -> Other':<30}", meta['company_grouping']['rare_companies'])
print()
print(pipeline)

Model file:    laptop_price_model.joblib (2097 KB)
Metadata file: model_metadata.json
model_name                     laptop-price-xgboost
model_version                  1.0.0
training_date                  2026-09-30
target_transformation          log
price_inverse_transformation   exp
training observations          1273
rare companies -> Other        ['Chuwi', 'Fujitsu', 'Google', 'Huawei', 'LG', 'Mediacom', 'Microsoft', 'Razer', 'Samsung', 'Vero', 'Xiaomi']

Pipeline(steps=[('company_grouper', CompanyGrouper()),
                ('preprocess',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['TypeName', 'Cpu brand',
                                                   'os', 'cpu_t

In [3]:
# Metrics from the locked notebook-05 protocol (BEFORE the final refit - not scores of the refit model itself)
ev = meta['evaluation_before_final_refit']
print(ev['note'], '\n')
print(pd.DataFrame([{**{k: v for k, v in ev['holdout'].items() if not isinstance(v, list)}}]).T.rename(columns={0: 'holdout (258 rows)'}))
print()
print(pd.DataFrame([ev['repeated_cv']]).T.rename(columns={0: 'repeated CV'}))

These metrics come from the locked evaluation BEFORE the final refit. The saved production model was refit on all 1,273 rows and therefore has no independent test score of its own; these numbers are the best estimate of its performance on new laptops. 

                    holdout (258 rows)
r2_log_price                  0.915952
r2_price                      0.823300
mae_rupees                 8700.476900
rmse_rupees               15275.053300
mape_percent                 13.916000
median_ape_percent           12.220500
mae_log_price                 0.136700

                                                    repeated CV
scheme             GroupKFold 5 folds x 5 repeats on the 1,0...
r2_log_price_mean                                        0.8974
r2_log_price_std                                         0.0138
r2_price_mean                                            0.8203
r2_price_std                                             0.0481
mae_rupees_mean                                  

## 2. Worked example: human-readable input -> engineered features -> price

In [4]:
example = {
    'company': 'Dell',
    'type_name': 'Notebook',
    'ram_gb': 8,
    'weight_kg': 2.2,
    'inches': 15.6,
    'resolution_width': 1920,
    'resolution_height': 1080,
    'touchscreen': False,
    'ips': False,
    'cpu': 'Intel Core i5 8250U 1.6GHz',
    'gpu': 'Nvidia GeForce MX150',
    'os': 'Windows 10',
    'ssd_gb': 256,
    'hdd_gb': 1000,
}
result = predict_laptop_price(example)
print('Engineered model features:')
for k, v in result['features'].items():
    print(f'  {k:<16} {v}')
print()
print(f"Predicted price (unrounded): Rs {result['predicted_price']:,.4f}")
print(f"Predicted price (display):   Rs {result['predicted_price_rounded']:,}")
print('Warnings:', result['warnings'])

Engineered model features:
  TypeName         Notebook
  Cpu brand        Intel Core i5
  os               Windows
  cpu_type         Low-power (U)
  cpu_gen          Gen 8
  gpu_tier         Nvidia other (MX/GT/9xxMX)
  Company_grouped  Dell
  Ram              8
  Weight           2.2
  Touchscreen      0
  Ips              0
  ppi              141.21199808219862
  SSD              256
  HDD              1000
  cpu_ghz          1.6
  Inches           15.6
  X_res            1920
  Y_res            1080

Predicted price (unrounded): Rs 47,971.1755
Predicted price (display):   Rs 48,000
Warnings: []


In [5]:
# The same interface on a laptop with unseen values: no crash, clear warnings
odd = {**example, 'company': 'Framework', 'cpu': 'Apple M2 3.5GHz', 'gpu': 'Nvidia GeForce RTX 4060', 'ram_gb': 128}
r = predict_laptop_price(odd)
print(f"Rs {r['predicted_price_rounded']:,}")
for w in r['warnings']:
    print(' -', w)

Rs 75,800
 - Company 'Framework' was not in the training data; it is treated as 'Other'.
 - CPU 'Apple M2 3.5GHz' was not in the training data; derived brand = 'AMD Processor'. Note: non-Intel CPUs are always treated as 'AMD Processor'.
 - GPU 'Nvidia GeForce RTX 4060' was not in the training data; its tier is derived from the name.
 - ram_gb=128 is outside the training range [2, 64]; the prediction is an extrapolation.


## 3. Sanity check on representative laptops
⚠️ **In-sample only.** All of these rows are part of the 1,273 rows the final model was trained on, so these errors are
**not** a measure of test performance (see the locked metrics above for that). The purpose is only to confirm that the
end-to-end user path (specs dict -> `predict_laptop_price`) gives sensible prices across very different laptops.

Each row is chosen deterministically: the laptop closest to the median price of its category.

In [6]:
raw = load_clean_raw()
integrated = raw['Gpu'].str.startswith('Intel')
categories = {
    'Cheap notebook (integrated)':     (raw['TypeName'] == 'Notebook') & (raw['Price'] < 25000) & integrated,
    'Mid-range notebook (discrete)':   (raw['TypeName'] == 'Notebook') & raw['Price'].between(40000, 60000) & ~integrated,
    'Ultrabook (Windows, integrated)': (raw['TypeName'] == 'Ultrabook') & raw['OpSys'].str.startswith('Windows') & integrated,
    'Gaming laptop (GTX)':             (raw['TypeName'] == 'Gaming') & raw['Gpu'].str.contains('GTX'),
    'Workstation (Quadro)':            (raw['TypeName'] == 'Workstation') & raw['Gpu'].str.contains('Quadro'),
    'Mac':                             raw['Company'] == 'Apple',
    '2-in-1 touchscreen':              (raw['TypeName'] == '2 in 1 Convertible') & raw['ScreenResolution'].str.contains('Touchscreen'),
    'AMD CPU laptop':                  raw['Cpu'].str.startswith('AMD'),
    'Linux / No OS':                   raw['OpSys'].isin(['Linux', 'No OS']),
    'Chromebook / Netbook':            (raw['OpSys'] == 'Chrome OS') | (raw['TypeName'] == 'Netbook'),
    'Very expensive (>150k)':          raw['Price'] > 150000,
}

def to_user_specs(row):
    spec = specs_from_raw(row.to_frame().T).iloc[0].to_dict()
    spec.pop('cpu_ghz')                                    # derived from the cpu text, as a website user would send
    return {k: (v.item() if hasattr(v, 'item') else v) for k, v in spec.items()}

rows = []
for label, mask in categories.items():
    group = raw[mask]
    idx = (group['Price'] - group['Price'].median()).abs().idxmin()
    row = raw.loc[idx]
    res = predict_laptop_price(to_user_specs(row))
    rows.append({'Category': label, 'Company': row['Company'], 'TypeName': row['TypeName'], 'Cpu': row['Cpu'],
                 'Gpu': row['Gpu'], 'Ram': row['Ram'], 'OpSys': row['OpSys'],
                 'Actual': round(row['Price']), 'Predicted': res['predicted_price_rounded'],
                 'Abs error': round(abs(res['predicted_price'] - row['Price'])),
                 'Pct error': round((row['Price'] - res['predicted_price']) / row['Price'] * 100, 1)})
sanity = pd.DataFrame(rows)
sanity

,Category,Company,TypeName,Cpu,Gpu,Ram,OpSys,Actual,Predicted,Abs error,Pct error
0,Cheap notebook (integrated),Dell,Notebook,Intel Core i3 6006U 2.0GHz,Intel HD Graphics 520,4GB,Linux,19128,21400,2276,-11.9
1,Mid-range notebook (discrete),Dell,Notebook,Intel Core i7 7500U 2.7GHz,Nvidia GeForce GT 940MX,8GB,Linux,47686,48300,641,-1.3
2,"Ultrabook (Windows, integrated)",Dell,Ultrabook,Intel Core i5 8250U 1.6GHz,Intel UHD Graphics 620,8GB,Windows 10,85195,85200,40,0.0
3,Gaming laptop (GTX),MSI,Gaming,Intel Core i7 7700HQ 2.8GHz,Nvidia GeForce GTX 1060,8GB,Windows 10,79215,71600,7597,9.6
4,Workstation (Quadro),Dell,Workstation,Intel Core i7 6820HQ 2.7GHz,Nvidia Quadro M1200,8GB,Windows 10,113753,114100,341,-0.3
5,Mac,Apple,Ultrabook,Intel Core i5 2.3GHz,Intel Iris Plus Graphics 640,8GB,macOS,71379,72900,1480,-2.1
6,2-in-1 touchscreen,Dell,2 in 1 Convertible,Intel Core i7 8550U 1.8GHz,Intel UHD Graphics 620,16GB,Windows 10,68145,68600,464,-0.7
7,AMD CPU laptop,Asus,Notebook,AMD A9-Series 9420 3GHz,AMD Radeon R5 M420,4GB,Windows 10,24988,22900,2061,8.2
8,Linux / No OS,Dell,Notebook,Intel Core i5 6200U 2.3GHz,Intel HD Graphics 520,4GB,Linux,27899,28500,557,-2.0
9,Chromebook / Netbook,HP,Netbook,Intel Celeron Dual Core N3060 1.6GHz,Intel HD Graphics 400,4GB,Chrome OS,20513,18700,1836,9.0


In [7]:
print('In-sample median absolute % error on these rows:', sanity['Pct error'].abs().median(), '%  (NOT a test score)')

In-sample median absolute % error on these rows: 2.1 %  (NOT a test score)


## 4. Artifact independence
Copy **only** `src/` and `artifacts/` into an empty temporary folder, start a brand-new Python process there and predict.
No notebook, CSV or notebook variable is available to that process.

In [8]:
with tempfile.TemporaryDirectory() as tmp:
    shutil.copytree('src', Path(tmp) / 'src', ignore=shutil.ignore_patterns('__pycache__'))
    shutil.copytree('artifacts', Path(tmp) / 'artifacts')
    print('Folder contents:', sorted(p.name for p in Path(tmp).rglob('*') if '__pycache__' not in p.parts))
    script = ('import json, sys; from src.predict import predict_laptop_price; '
              'print(json.dumps(predict_laptop_price(json.loads(sys.argv[1]))))')
    out = subprocess.run([sys.executable, '-c', script, json.dumps(example)], cwd=tmp, capture_output=True, text=True)
    assert out.returncode == 0, out.stderr
fresh = json.loads(out.stdout)
print('Fresh-process prediction: Rs', f"{fresh['predicted_price']:,.4f}")
print('This notebook:            Rs', f"{result['predicted_price']:,.4f}")
print('Identical:', fresh['predicted_price'] == result['predicted_price'])

Folder contents: ['__init__.py', 'artifacts', 'features.py', 'laptop_price_model.joblib', 'model_metadata.json', 'predict.py', 'src', 'train.py']


Fresh-process prediction: Rs 47,971.1755
This notebook:            Rs 47,971.1755
Identical: True
